# step3 — 준수 개수 확장(8..5) + 지침 snake 방향 (RQ1 headroom·절벽)

**대응 RQ:** RQ1 — 선행 위반이 준수율을 낮추는가. stepA의 개수 축(4..0)을 넓히고 지침 snake 방향을 추가한다.

**3그룹(병렬, 브랜치 1개씩)**
- `camel-8to5` (`step3/camel-8to5`): 준수 예시 **다수(8/7/6/5)** 로도 camel-on-Python 바닥이 유지되나. (camel 4..0은 stepA에 있어 재실행 안 함 → 분석 때 이어 붙임)
- `snake-4to0` (`step3/snake-4to0`): snake는 Python 관습 정렬이라 baseline 높음(**headroom**). 준수(snake)를 4→0으로 줄이면 준수율이 **절벽**으로 떨어지나 = 우리 하네스 첫 절벽 재현 여부.
- `snake-8to5` (`step3/snake-8to5`): snake headroom 위쪽 구간.

**바꾼 것은 개수 축 + 지침 방향뿐**, 나머지는 stepA와 동일(CLONE 12, 약한 어조, greedy, seed 20, 개입 없음).

**병렬:** 세 Colab 세션에서 셀 3의 `GROUP`만 각 브랜치에 맞게 두면 된다(노트북은 모든 브랜치에서 동일 — 머지 충돌 없음). 결과는 조건이 파일명에 드러나 충돌하지 않는다(§6).

> **메모리(무료 T4):** Qwen2.5-Coder-3B fp16 ≈ 6.2GB. 그룹당 4~5개수 × 20 seed × 3턴 ≈ 15~30분. **재개 가능**(저장된 조건 건너뜀).


In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)


In [ ]:
# GROUP 선택 (병렬 세션마다 여기만 바꾼다) → 브랜치·조건이 여기서 결정됨
GROUP = 'camel-8to5'   # 'camel-8to5' | 'snake-4to0' | 'snake-8to5'
BRANCH = f'step3/{GROUP}'

# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin {BRANCH}
!git checkout {BRANCH}
!git pull --quiet origin {BRANCH}
!pip install -e . -q
import sys; sys.path.insert(0, 'src')


In [ ]:
# 조건 설정 — GROUP으로 방향·개수 결정 (로직은 harness가 수행)
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)
MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
GROUPS = {
    'camel-8to5': (Notation.CAMEL, [8, 7, 6, 5]),
    'snake-4to0': (Notation.SNAKE, [4, 3, 2, 1, 0]),
    'snake-8to5': (Notation.SNAKE, [8, 7, 6, 5]),
}
TARGET, N_COMPLIANT = GROUPS[GROUP]
SEEDS = list(range(20))
def make(n, s):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=n, composition=Composition.CLONE),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=TARGET),
        seed=s)
conditions = [make(n, s) for n in N_COMPLIANT for s in SEEDS]
PREDICTIONS = {
    'camel-8to5': '준수 다수(8..5)로도 camel-on-Python 바닥 유지 가능(다수결로 다소 상승 가능). stepA 4..0과 이어 곡선 완성.',
    'snake-4to0': 'snake headroom. 준수(snake) 4->0로 줄이면 위반(camel) 증가 → 준수율 절벽 하락 예상 = 하네스 첫 절벽 재현 여부.',
    'snake-8to5': 'snake headroom 위쪽. 8..5는 높게 유지 예상.',
}
PREDICTION = PREDICTIONS[GROUP]
print(GROUP, '|', len(conditions), '조건 =', len(N_COMPLIANT), 'x', len(SEEDS))


In [ ]:
# 실행 — 조건별 순차 생성 + 즉시 저장(재개 가능)
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
handle = load_model(MODEL)
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info())
new = skipped = 0
for c in conditions:
    if result_path(c, step='step3').exists():
        skipped += 1; continue
    out = run(c, handle=handle)
    save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                             step='step3', rq='RQ1', prediction=PREDICTION))
    new += 1
    if new % 10 == 0:
        print(f'생성 {new} / 건너뜀 {skipped} / 총 {len(conditions)}')
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')


In [ ]:
# 결과 로드 — results/step3/
from harness import result_path
from harness.results import load_result
records = [load_result(result_path(c, step='step3')) for c in conditions]
print('로드:', len(records), '건 → results/step3/', '| GROUP =', GROUP)


In [ ]:
# 요약 — 이 그룹 준수율 곡선 + 자기증폭
import pandas as pd, matplotlib.pyplot as plt
rows = [{'n_compliant': r.condition.preceding.n_compliant,
         'compliant': r.metrics.extra['first_compliant'],
         'first_violated': r.metrics.extra['first_violated'],
         'subseq_viol': r.metrics.extra['subsequent_violation_rate']} for r in records]
df = pd.DataFrame(rows)
rate = df.groupby('n_compliant')['compliant'].mean().reindex(N_COMPLIANT)
print(GROUP, '준수율(첫 함수):'); print(rate.round(3))
amp = df[df.first_violated].groupby('n_compliant')['subseq_viol'].mean()
print('\n자기증폭(첫 위반 조건에서 이후 위반 비율):'); print(amp.round(3))
plt.figure(figsize=(5,3))
plt.plot([str(n) for n in N_COMPLIANT], rate.values, marker='o')
plt.xlabel('n_compliant'); plt.ylabel('준수율(첫 함수)')
plt.title(f'step3 {GROUP}'); plt.ylim(-0.02, 1.02); plt.grid(True, alpha=.3); plt.show()
